# 02 — MWL exploration

Explore perceived Mental Workload before testing physiological associations. All analyses are descriptive and retain the repeated-measures structure.

## 1. Imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import iqr

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

## 2. Paths and configuration

In [ ]:
SAVE_FIGURES = True
SAVE_TABLES = True
INCLUDE_IMPUTED_MWL = True
PHASE_ORDER = ["pre_test", "test_1", "test_2", "test_3", "evaluation"]
GROUP_ORDER = ["NoHA", "Haptic"]

def find_repository_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "outputs/final_features/physiology_mwl_analysis_dataset.csv").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root from the current directory")

REPO_ROOT = find_repository_root()
INPUT_PATH = REPO_ROOT / "outputs/final_features/physiology_mwl_analysis_dataset.csv"
FIGURE_DIR = REPO_ROOT / "postprocessing/outputs/figures"
TABLE_DIR = REPO_ROOT / "postprocessing/outputs/tables"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
print("INCLUDE_IMPUTED_MWL =", INCLUDE_IMPUTED_MWL)

## 3. Load and validate

In [ ]:
all_data = pd.read_csv(INPUT_PATH)
data = all_data.copy() if INCLUDE_IMPUTED_MWL else all_data.loc[all_data.mwl_source.eq("observed")].copy()
data["phase"] = pd.Categorical(data.phase, categories=PHASE_ORDER, ordered=True)
assert all_data.mwl_value.notna().all()
assert not all_data.duplicated(["participant_id", "phase", "block_index"]).any()
assert set(all_data.mwl_source) <= {"observed", "imputed_previous"}
print(f"Using {len(data)} of {len(all_data)} MWL observations from {data.participant_id.nunique()} participants.")

## 4. Approved imputation and sensitivity configuration

In [ ]:
imputed_observation = all_data.loc[all_data.mwl_source.eq("imputed_previous"), ["participant_id", "group", "phase", "block_index", "mwl_value", "mwl_source"]]
print("Approved imputed observation:\n", imputed_observation.to_string(index=False))
if SAVE_TABLES:
    imputed_observation.to_csv(TABLE_DIR / "02_imputed_mwl_observation.csv", index=False)

## 5. Descriptive statistics

In [ ]:
def describe_mwl(frame, group_columns=None):
    group_columns = [] if group_columns is None else list(group_columns)
    grouped = [((), frame)] if not group_columns else frame.groupby(group_columns, observed=True, sort=False)
    rows = []
    for key, subset in grouped:
        key = key if isinstance(key, tuple) else (key,)
        values = subset.mwl_value.dropna().astype(float)
        row = dict(zip(group_columns, key))
        row.update({
            "N": len(values), "mean": values.mean(), "SD": values.std(ddof=1),
            "median": values.median(), "IQR": iqr(values, rng=(25, 75), nan_policy="omit"),
            "min": values.min(), "max": values.max(),
        })
        rows.append(row)
    return pd.DataFrame(rows)

overall_stats = describe_mwl(data)
group_stats = describe_mwl(data, ["group"])
phase_stats = describe_mwl(data, ["phase"])
group_phase_stats = describe_mwl(data, ["group", "phase"])
print("Overall:\n", overall_stats.to_string(index=False))
print("\nBy group:\n", group_stats.to_string(index=False))
print("\nBy phase:\n", phase_stats.to_string(index=False))
print("\nBy group × phase:\n", group_phase_stats.to_string(index=False))
if SAVE_TABLES:
    overall_stats.to_csv(TABLE_DIR / "02_mwl_descriptive_overall.csv", index=False)
    group_stats.to_csv(TABLE_DIR / "02_mwl_descriptive_by_group.csv", index=False)
    phase_stats.to_csv(TABLE_DIR / "02_mwl_descriptive_by_phase.csv", index=False)
    group_phase_stats.to_csv(TABLE_DIR / "02_mwl_descriptive_by_group_phase.csv", index=False)

## 6. MWL distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
bins = np.arange(np.floor(data.mwl_value.min()) - 0.5, np.ceil(data.mwl_value.max()) + 1.5, 1)
for group in GROUP_ORDER:
    values = data.loc[data.group.eq(group), "mwl_value"]
    axes[0].hist(values, bins=bins, alpha=0.55, label=group, edgecolor="white")
axes[0].set(xlabel="MWL rating", ylabel="Observations", title="MWL distribution by group")
axes[0].legend()

positions, box_values, colors = [], [], []
for phase_index, phase in enumerate(PHASE_ORDER, 1):
    for group_index, group in enumerate(GROUP_ORDER):
        values = data.loc[data.phase.eq(phase) & data.group.eq(group), "mwl_value"].dropna()
        if len(values):
            positions.append(phase_index + (-0.18 if group_index == 0 else 0.18))
            box_values.append(values)
            colors.append("tab:blue" if group == "NoHA" else "tab:orange")
boxes = axes[1].boxplot(box_values, positions=positions, widths=0.3, patch_artist=True)
for patch, color in zip(boxes["boxes"], colors): patch.set_facecolor(color); patch.set_alpha(0.55)
axes[1].set_xticks(range(1, len(PHASE_ORDER) + 1), PHASE_ORDER, rotation=25)
axes[1].set(xlabel="Phase", ylabel="MWL rating", title="MWL by phase and group")
axes[1].grid(axis="y", alpha=0.2)
fig.tight_layout()
if SAVE_FIGURES:
    fig.savefig(FIGURE_DIR / "02_mwl_distribution.png", dpi=160)
plt.show()

## 7. MWL progression with block resolution

Test phases retain their block index. Pre-Test and Evaluation each have one block.

In [ ]:
def stage_label(row):
    return row["phase"] if row["phase"] in ("pre_test", "evaluation") else f"{row['phase']}_b{int(row['block_index'])}"

stage_order = ["pre_test", "test_1_b1", "test_1_b2", "test_2_b1", "test_2_b2", "test_2_b3", "test_3_b1", "test_3_b2", "test_3_b3", "evaluation"]
data["stage"] = data.apply(stage_label, axis=1)
data["stage"] = pd.Categorical(data.stage, categories=stage_order, ordered=True)
progression = data.groupby(["group", "stage"], observed=True).mwl_value.agg(["count", "mean", "std"]).reset_index()
progression["SEM"] = progression["std"] / np.sqrt(progression["count"])
print(progression.to_string(index=False))
if SAVE_TABLES:
    progression.to_csv(TABLE_DIR / "02_mwl_progression_by_group.csv", index=False)

fig, ax = plt.subplots(figsize=(12, 4.5))
for group, color in zip(GROUP_ORDER, ["tab:blue", "tab:orange"]):
    subset = progression[progression.group.eq(group)].set_index("stage").reindex(stage_order)
    x = np.arange(len(stage_order))
    ax.errorbar(x, subset["mean"], yerr=subset["SEM"], marker="o", capsize=3, label=group, color=color)
ax.set_xticks(np.arange(len(stage_order)), stage_order, rotation=35, ha="right")
ax.set(ylabel="Mean MWL rating", title="Descriptive MWL progression (mean ± SEM)")
ax.legend(); ax.grid(alpha=0.25)
fig.tight_layout()
if SAVE_FIGURES:
    fig.savefig(FIGURE_DIR / "02_mwl_progression_by_group.png", dpi=160)
plt.show()

## 8. Within-subject trajectories

Lines connect repeated observations from the same participant; they are not treated as independent.

In [ ]:
participant_stage = data.pivot_table(index=["participant_id", "group"], columns="stage", values="mwl_value", aggfunc="mean").reindex(columns=stage_order)
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(stage_order))
for (participant, group), row in participant_stage.iterrows():
    color = "tab:blue" if group == "NoHA" else "tab:orange"
    ax.plot(x, row.to_numpy(dtype=float), color=color, alpha=0.28, linewidth=1)
for group, color in zip(GROUP_ORDER, ["tab:blue", "tab:orange"]):
    mean_line = participant_stage.xs(group, level="group").mean(axis=0)
    ax.plot(x, mean_line, color=color, linewidth=3, marker="o", label=f"{group} participant mean")
ax.set_xticks(x, stage_order, rotation=35, ha="right")
ax.set(ylabel="Participant mean MWL", title="Within-subject MWL trajectories")
ax.legend(); ax.grid(alpha=0.2)
fig.tight_layout()
if SAVE_FIGURES:
    fig.savefig(FIGURE_DIR / "02_mwl_participant_trajectories.png", dpi=160)
plt.show()

## 9. Descriptive within- versus between-subject variability

A one-way random-intercept method-of-moments decomposition is used as an ICC-style descriptive summary. It is not the final inferential repeated-measures model.

In [ ]:
participant_groups = list(data.groupby("participant_id", observed=True).mwl_value)
N = len(data); k = len(participant_groups); grand_mean = data.mwl_value.mean()
ss_between = sum(len(values) * (values.mean() - grand_mean) ** 2 for _, values in participant_groups)
ss_within = sum(((values - values.mean()) ** 2).sum() for _, values in participant_groups)
ms_between = ss_between / (k - 1)
ms_within = ss_within / (N - k)
group_sizes = np.array([len(values) for _, values in participant_groups], dtype=float)
effective_n = (N - (group_sizes ** 2).sum() / N) / (k - 1)
between_variance = max((ms_between - ms_within) / effective_n, 0.0)
within_variance = ms_within
icc_descriptive = between_variance / (between_variance + within_variance) if between_variance + within_variance else np.nan
variance_decomposition = pd.DataFrame([{
    "N_observations": N, "N_participants": k, "effective_cluster_size": effective_n,
    "between_participant_variance": between_variance,
    "within_participant_variance": within_variance,
    "descriptive_ICC": icc_descriptive,
}])
print(variance_decomposition.to_string(index=False))
if SAVE_TABLES:
    variance_decomposition.to_csv(TABLE_DIR / "02_mwl_variance_decomposition.csv", index=False)

## 10. Imputation sensitivity

The summaries below make the effect of the single approved imputation visible without fitting any model.

In [ ]:
sensitivity_rows = []
for label, subset in [("include_imputed", all_data), ("observed_only", all_data.loc[all_data.mwl_source.eq("observed")])]:
    values = subset.mwl_value
    sensitivity_rows.append({"analysis": label, "N": len(values), "mean": values.mean(), "SD": values.std(ddof=1), "median": values.median(), "IQR": iqr(values)})
sensitivity = pd.DataFrame(sensitivity_rows)
print(sensitivity.to_string(index=False))
if SAVE_TABLES:
    sensitivity.to_csv(TABLE_DIR / "02_mwl_imputation_sensitivity.csv", index=False)

## 11. Exploratory summary

In [ ]:
print("MWL EXPLORATION SUMMARY")
print(f"- observations analyzed: {len(data)}")
print(f"- participants: {data.participant_id.nunique()}")
print(f"- groups: {', '.join(map(str, data.group.unique()))}")
print(f"- observed MWL rows: {int(data.mwl_source.eq('observed').sum())}")
print(f"- imputed MWL rows included: {int(data.mwl_source.eq('imputed_previous').sum())}")
print(f"- descriptive ICC: {icc_descriptive:.4f}")
print("- no physiological associations or inferential models were fitted")